In [3]:
import polars as pl
import numpy as np
import time
from gensim.corpora import Dictionary
from gensim.models import LdaMulticore
from gensim.models import Phrases
from gensim.models.phrases import Phraser
from gensim.parsing.preprocessing import STOPWORDS
import joblib
from tqdm.notebook import tqdm
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
import os
from concurrent.futures import ProcessPoolExecutor, as_completed
import gc

yelp_stopwords = [
    "food", "good", "place", "service", "restaurant", "great", "time", 
    "go", "back", "really", "just", "like", "get", "one", "would", 
    "ive", "even", "also", "always", "got", "came", "went", "us", "im", "much"
]
stopwords_list = list(STOPWORDS) + [""] + yelp_stopwords

In [15]:
def train_all_lda(input_csv: str, categories: dict):
    for key, label in categories.items():
        print(f'\n{'='*50}\n[{label}] Entrenando LDA...\n{'='*50}')

        df = (
            pl.scan_csv(input_csv)
            .filter(pl.col('categories').str.contains(label))
            .collect()
        )
        print(f'[{label}] {len(df):,} reviews cargadas')

        train_lda_for_category_from_df(df, key, label)
        del df


def train_lda_for_category_from_df(df: pl.DataFrame, category_key: str, category_label: str):
    model_dir = f'../data/models/{category_key}/lda'
    os.makedirs(model_dir, exist_ok=True)

    sample_size = min(300_000, len(df))
    df = df.sample(n=sample_size, seed=42)

    df = df.with_columns(
        pl.col('text')
        .str.replace_all(r'[^a-zA-Z\s]', '')
        .str.to_lowercase()
        .str.split(' ')
        .list.set_difference(stopwords_list)
        .alias('tokens')
    )
    tokenized_docs = df['tokens'].to_list()
    del df

    bigram_detector = Phrases(tokenized_docs, min_count=10, threshold=20)
    bigram_model    = Phraser(bigram_detector)
    tokenized_docs  = [bigram_model[doc] for doc in tokenized_docs]

    dictionary = Dictionary(tokenized_docs)
    no_below = 20 if sample_size > 100_000 else 5
    dictionary.filter_extremes(no_below=no_below, no_above=0.5, keep_n=20_000)
    corpus = [dictionary.doc2bow(t) for t in tokenized_docs]
    del tokenized_docs

    lda = LdaMulticore(
        corpus=corpus,
        num_topics=10,
        id2word=dictionary,
        workers=max(1, os.cpu_count() - 1),
        passes=3,
        chunksize=2_000,
        random_state=42,
    )

    lda.save(f'{model_dir}/lda_model.gensim')
    dictionary.save(f'{model_dir}/dictionary.dict')
    bigram_model.save(f'{model_dir}/bigram_model.pkl')
    print(f'[{category_label}] LDA guardado -> {model_dir}')

In [16]:
def train_all_bertopic(input_csv: str, categories: dict):
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

    for key, label in categories.items():
        model_dir = f'../data/models/{key}/bertopic'
        os.makedirs(model_dir, exist_ok=True)

        df = (
            pl.scan_csv(input_csv)
            .filter(pl.col('categories').str.contains(label))
            .collect()
        )
        sample_size = min(300_000, len(df))
        docs = df.sample(n=sample_size, seed=42)['text'].to_list()
        del df

        print(f'[{label}] Embedding {len(docs):,} docs...')
        embeddings = embedding_model.encode(
            docs,
            batch_size=512,
            show_progress_bar=True,
            convert_to_numpy=True,
        )

        min_df = 10 if sample_size > 100_000 else 3
        vectorizer_model = CountVectorizer(
            stop_words=stopwords_list,
            ngram_range=(2, 3),
            min_df=min_df,
        )
        topic_model = BERTopic(
            calculate_probabilities=False,
            verbose=True,
            vectorizer_model=vectorizer_model,
        )
        topic_model.fit_transform(docs, embeddings=embeddings)
        topic_model.save(f'{model_dir}/bertopic_model.pkl', serialization='pickle')
        print(f'[{label}] BERTopic guardado -> {model_dir}')

        del embeddings, docs, topic_model

    del embedding_model

In [ ]:
def load_category_models(category_key, embedding_model):
    lda_dir = f'../data/models/{category_key}/lda'
    bert_dir = f'../data/models/{category_key}/bertopic'
    
    lda = LdaMulticore.load(f'{lda_dir}/lda_model.gensim')
    dictionary = Dictionary.load(f'{lda_dir}/dictionary.dict')
    bigram_model = Phraser.load(f'{lda_dir}/bigram_model.pkl')
    
    bertopic = BERTopic.load(f'{bert_dir}/bertopic_model.pkl', embedding_model=embedding_model)
    
    lda_topic_words = {}
    for i in range(lda.num_topics):
        words = lda.show_topic(i, topn=3)
        lda_topic_words[i] = f'{words[0][0]}_{words[1][0]}_{words[2][0]}'
        
    bert_labels = bertopic.topic_labels_
    
    return lda, dictionary, bigram_model, bertopic, lda_topic_words, bert_labels

def run_inference_all_categories(input_csv: str, out_dir: str, categories: dict):
    os.makedirs(out_dir, exist_ok=True)

    print('\nCargando SentenceTransformer base para inferencia (se reutiliza)...')
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

    start_time = time.time()

    for key, label in categories.items():
        print(f'\n[{label}] Procesando categoría...')

        print(f'[{label}] Cargando modelos de disco...')
        lda, dictionary, bigram_model, bertopic, lda_topic_words, bert_labels = load_category_models(key, embedding_model)

        df = (
            pl.scan_csv(input_csv)
            .filter(pl.col('categories').str.contains(label))
            .collect()
        )
        print(f'[{label}] {len(df):,} reviews cargadas')

        print(f'[{label}] LDA sobre {len(df):,} docs...')

        df_tok = df.with_columns(
            pl.col('text')
            .str.replace_all(r'[^a-zA-Z\s]', '')
            .str.to_lowercase()
            .str.split(' ')
            .list.set_difference(stopwords_list)
            .alias('tokens')
        )
        bigramed = [bigram_model[doc] for doc in df_tok['tokens'].to_list()]
        del df_tok

        bow   = [dictionary.doc2bow(t) for t in bigramed]
        dists = lda[bow]
        del bigramed, bow
    
        dominant_topics, topic_probs = [], []
        for dist in dists:
            if dist:
                best_id, best_prob = max(dist, key=lambda x: x[1])
                dominant_topics.append(lda_topic_words[best_id])
                topic_probs.append(best_prob)
            else:
                dominant_topics.append('none')
                topic_probs.append(0.0)
        del dists

        print(f'[{label}] BERTopic transform sobre {len(df):,} docs...')
        bert_topics, _ = bertopic.transform(df['text'].to_list())
        bert_labels_col = [bert_labels.get(t, f'topic_{t}') for t in bert_topics]

        out_path = os.path.join(out_dir, f'yelp_topics_{key}.csv')
        df.with_columns([
            pl.Series('lda_dominant_topic',      dominant_topics),
            pl.Series('lda_topic_probability',   topic_probs),
            pl.Series('bertopic_topic',           bert_topics),
            pl.Series('bertopic_dominant_topic',  bert_labels_col),
        ]).write_csv(out_path)

        print(f'[{label}]Guardado -> {out_path}')

        del df, dominant_topics, topic_probs, bert_topics, bert_labels_col
        del lda, dictionary, bigram_model, bertopic, lda_topic_words, bert_labels
        
        gc.collect()

    print(f'\nInferencia completada en {(time.time() - start_time) / 60:.2f} min')

In [30]:
csv_reviews_with_categories = '../data/csv/yelp_reviews_with_business.csv'
csv_output_dir = '../results/topic_modeling'

CATEGORIES = {
    "fast_food":   "Fast Food",
    "steakhouses": "Steakhouses",
    "burgers":     "Burgers",
    "pubs":        "Pubs",
    "mexican":     "Mexican",
}

run_inference_all_categories(csv_reviews_with_categories, csv_output_dir, CATEGORIES)


Cargando SentenceTransformer base para inferencia (se reutiliza)...


Loading weights: 100%|█████████████████████████████████████████████| 103/103 [00:00<00:00, 766.45it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[Fast Food] Procesando categoría...
[Fast Food] Cargando modelos de disco...
[Fast Food] 233,008 reviews cargadas
[Fast Food] BERTopic transform sobre 233,008 docs...


Batches: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 7282/7282 [03:40<00:00, 33.01it/s]
2026-04-19 23:04:38,122 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-04-19 23:06:40,641 - BERTopic - Dimensionality - Completed ✓
2026-04-19 23:06:40,641 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-04-19 23:07:24,137 - BERTopic - Cluster - Completed ✓


[Fast Food] ✓ Guardado -> ../results/topic_modeling/yelp_topics_fast_food.csv

[Steakhouses] Procesando categoría...
[Steakhouses] Cargando modelos de disco...
[Steakhouses] 240,040 reviews cargadas
[Steakhouses] BERTopic transform sobre 240,040 docs...


...

[Burgers] ✓ Guardado -> ../results/topic_modeling/yelp_topics_burgers.csv

[Pubs] Procesando categoría...
[Pubs] Cargando modelos de disco...
[Pubs] 218,891 reviews cargadas
[Pubs] BERTopic transform sobre 218,891 docs...


Batches: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 6841/6841 [03:25<00:00, 33.27it/s]
2026-04-19 23:39:35,428 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-04-19 23:43:06,363 - BERTopic - Dimensionality - Completed ✓
2026-04-19 23:43:06,393 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-04-19 23:44:06,648 - BERTopic - Cluster - Completed ✓


[Pubs] ✓ Guardado -> ../results/topic_modeling/yelp_topics_pubs.csv

[Mexican] Procesando categoría...
[Mexican] Cargando modelos de disco...
[Mexican] 432,248 reviews cargadas
[Mexican] BERTopic transform sobre 432,248 docs...


Batches: 100%|███████████████████████████████████████████████████████████████████████████████████████████| 13508/13508 [06:57<00:00, 32.38it/s]
2026-04-19 23:55:19,095 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-04-20 00:00:55,414 - BERTopic - Dimensionality - Completed ✓
2026-04-20 00:00:55,415 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-04-20 00:04:13,714 - BERTopic - Cluster - Completed ✓


[Mexican] ✓ Guardado -> ../results/topic_modeling/yelp_topics_mexican.csv

Inferencia completada en 66.02 min
